In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import wfdb
import keras
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, BatchNormalization, ReLU, Conv1DTranspose, Concatenate
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

In [ ]:
root_dir="./ptb_xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1/records500"

file_paths=[]
for dirpath, dirnames, filenames in os.walk(root_dir):
    for file in filenames:
        if file.endswith(".dat"):
            file_paths.append(os.path.join(dirpath, file))
file_paths = np.asarray(file_paths)

In [ ]:
file_paths.shape

In [ ]:
# 70% train, 30% temp -> temp split into 10% val / 20% test
train_paths, temp_paths = train_test_split(
    file_paths, test_size=0.30, random_state=42, shuffle=True
)
val_paths, test_paths = train_test_split(
    temp_paths, test_size=2/3, random_state=42, shuffle=True
)

print(f"Train: {len(train_paths)} files")
print(f"Val:   {len(val_paths)} files")
print(f"Test:  {len(test_paths)} files")

In [ ]:
def load_challenge_data(filename):
    x , _= wfdb.rdsamp(os.path.splitext(filename)[0])
    return x

In [ ]:
load_challenge_data(train_paths[0])

In [ ]:
lead_labels = [
    'I',  'II', 'III',
    'aVR','aVL','aVF', 
    'V1', 'V2', 'V3', 
    'V4', 'V5', 'V6', 
    'V1', 'II', 'V5'
]

In [ ]:
quartile = 1250

In [ ]:
# ---- Batch Generators ----
def batch_generator(batch_size, gen_ecg):
    """
    Yields batches of (features, labels).
    """
    batch_input = np.zeros((batch_size, 5000, 15))
    batch_output = np.zeros((batch_size, 5000, 12))

    while True:
        for i in range(batch_size):
            batch_output[i], batch_input[i] = next(gen_ecg)

        yield batch_input, batch_output

In [ ]:
def generate_io(files, quartile):
    """
    Loads and preprocesses ECG data, looping over the files forever.
    """
    while True:
        for i in files:
            data = load_challenge_data(i)
            X_processed = process_ecg(data, quartile)
            yield np.expand_dims(data,0), X_processed

In [ ]:
# ---- ECG Preprocessing ----
def process_ecg(X_train, quartile):
    """
    Apply channel masking and zeroing according to quartile rules.
    """
    if len(X_train.shape) == 2:
        X_train = np.expand_dims(X_train, 0)
    T = X_train.shape[1]  # time length
    X_train_input = X_train[:,: ,[0,1,2,3,4,5,6,7,8,9,10,11,6,1,10]]

    # zero out chunks by quartiles
    X_train_input[:, int(quartile):,:3] = 0

    idx1 = np.r_[0:int(quartile), int(quartile*2):T]
    X_train_input[:, idx1,3:6] = 0

    idx2 = np.r_[:int(quartile*2), int(quartile*3):T]
    X_train_input[:, idx2, 6:9] = 0

    X_train_input[:, :int(quartile*3), 9:12] = 0

    return X_train_input

In [ ]:
test_gen = generate_io(train_paths, quartile)

In [ ]:
a,b = next(test_gen)

In [ ]:
a.shape

In [ ]:
# Self-Attention Layer
@keras.saving.register_keras_serializable()  # needed to load the saved model
class SelfAttention1D(layers.Layer):
    def __init__(self, channels, **kwargs):
        super(SelfAttention1D, self).__init__(**kwargs)
        self.channels = channels

        self.query_dense = layers.Dense(channels)
        self.key_dense = layers.Dense(channels)
        self.value_dense = layers.Dense(channels)
        self.softmax = layers.Softmax(axis=-1)

    def call(self, inputs):
        # QKV projections
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)

        # Attention weights
        score = tf.matmul(query, key, transpose_b=True)
        score = score / tf.sqrt(tf.cast(self.channels, tf.float32))
        weights = self.softmax(score)

        output = tf.matmul(weights, value)
        return output

    def get_config(self):
        config = super(SelfAttention1D, self).get_config()
        config.update({
            "channels": self.channels,
        })
        return config

In [ ]:
def build_unet_ecg(input_shape=(5000, 15)):
    # Input: 5000 time steps, 12 masked leads + full II, V1, V5
    inputs = Input(shape=input_shape)

    # --------------------- Encoder ---------------------
    # Block 1
    e1 = Conv1D(64, 3, padding='same')(inputs)
    e1 = BatchNormalization()(e1)
    e1 = ReLU()(e1)
    p1 = MaxPooling1D(2)(e1)  # Output: 2500

    # Block 2
    e2 = Conv1D(128, 3, padding='same')(p1)
    e2 = BatchNormalization()(e2)
    e2 = ReLU()(e2)
    p2 = MaxPooling1D(2)(e2)  # Output: 1250

    # Block 3
    e3 = Conv1D(256, 3, padding='same')(p2)
    e3 = BatchNormalization()(e3)
    e3 = ReLU()(e3)
    p3 = MaxPooling1D(2)(e3)  # Output: 625

    # Block 4 (Bottleneck)
    e4 = Conv1D(512, 3, padding='same')(p3)
    e4 = BatchNormalization()(e4)
    e4 = ReLU()(e4)
    e4 = SelfAttention1D(512)(e4)

    # --------------------- Decoder ---------------------
    # Block 1 (Upsample to 1250)
    d1 = Conv1DTranspose(256, 3, strides=2, padding='same')(e4)
    d1 = Concatenate()([d1, e3])  # Skip connection from e3
    d1 = Conv1D(256, 3, padding='same')(d1)
    d1 = BatchNormalization()(d1)
    d1 = ReLU()(d1)

    # Block 2 (Upsample to 2500)
    d2 = Conv1DTranspose(128, 3, strides=2, padding='same')(d1)
    d2 = Concatenate()([d2, e2])  # Skip connection from e2
    d2 = Conv1D(128, 3, padding='same')(d2)
    d2 = BatchNormalization()(d2)
    d2 = ReLU()(d2)

    # Block 3 (Upsample to 5000)
    d3 = Conv1DTranspose(64, 3, strides=2, padding='same')(d2)
    d3 = Concatenate()([d3, e1])  # Skip connection from e1
    d3 = Conv1D(64, 3, padding='same')(d3)
    d3 = BatchNormalization()(d3)
    d3 = ReLU()(d3)

    # Output Layer: all 12 leads
    outputs = Conv1D(12, 1, activation='linear')(d3)

    model = Model(inputs, outputs)
    return model

In [ ]:
model = build_unet_ecg(b.shape[1:])
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4),
    loss='mse',
    metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

model.summary()

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    print(gpu)

In [ ]:
model_path = "/kaggle/working/best_ecg_reconstruction_model.keras"

checkpoint = ModelCheckpoint(
    model_path,
    monitor='val_loss',
    verbose=1,
    save_best_only=True,
    mode='min'
)

history = model.fit(
    batch_generator(32, generate_io(train_paths,quartile)),
    steps_per_epoch=len(train_paths) // 32,
    validation_data = batch_generator(32, generate_io(val_paths,quartile)),
    validation_steps=len(val_paths) // 32,
    epochs=100,
    callbacks=[checkpoint]
)

best_model = load_model(
    model_path,
    custom_objects={"SelfAttention1D": SelfAttention1D}
)

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 5))
plt.plot(train_loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
test_data_out = []
test_data_in = []
for i in test_paths:
    output_data = load_challenge_data(i)
    input_data = process_ecg(output_data, quartile)[0]
    test_data_out.append(output_data)
    test_data_in.append(input_data)
test_data_out = np.asarray(test_data_out)
test_data_in = np.asarray(test_data_in)

In [ ]:
Y_pred = best_model.predict(test_data_in)

In [ ]:
np.savez("data.npz", original=test_data_out, reconstructed=Y_pred)

In [ ]:
# Compute Pearson Correlation Coefficient for each lead
pcc_scores = [pearsonr(test_data_out[:, :, i].flatten(), Y_pred[:, :, i].flatten())[0] for i in range(test_data_out.shape[2])]

print(f"Mean PCC across leads: {np.mean(pcc_scores):.4f}")

print("PCC pr lead:")
for num, score in enumerate(pcc_scores):
    print(lead_labels[num] + ": ",score )

In [ ]:
# Compute RMSE for each lead
rmse_scores = [np.sqrt(mean_squared_error(test_data_out[:, :, i].flatten(),
                                          Y_pred[:, :, i].flatten()))
               for i in range(test_data_out.shape[2])]

print(f"Mean RMSE across leads: {np.mean(rmse_scores):.4f}")
print("\nRMSE per lead:")
for num, score in enumerate(rmse_scores):
    print(f"{lead_labels[num]}: {score:.4f}")

In [ ]:
# Correction based on leads we know (II, V1, V5)
known_idx = [1, 6, 10]
Y_pred[:,:,known_idx] = test_data_out[:,:,known_idx]

In [ ]:
# Compute Pearson Correlation Coefficient for each lead
pcc_scores = [pearsonr(test_data_out[:, :, i].flatten(), Y_pred[:, :, i].flatten())[0] for i in range(test_data_out.shape[2])]

print(f"Mean PCC across leads: {np.mean(pcc_scores):.4f}")

print("PCC pr lead:")
for num, score in enumerate(pcc_scores):
    print(lead_labels[num] + ": ",score )

In [ ]:
# Compute RMSE for each lead
rmse_scores = [np.sqrt(mean_squared_error(test_data_out[:, :, i].flatten(), 
                                          Y_pred[:, :, i].flatten())) 
               for i in range(test_data_out.shape[2])]

print(f"Mean RMSE across leads: {np.mean(rmse_scores):.4f}")
print("\nRMSE per lead:")
for num, score in enumerate(rmse_scores):
    print(f"{lead_labels[num]}: {score:.4f}")

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(20, 10))
fig.suptitle("Test recording 0", fontsize=16)

cnt = 0
for i in range(4):
    for j in range(3):
        ax = axes[i, j]
        ax.set_title(lead_labels[cnt])
        ax.plot(Y_pred[0,:,cnt], label="Reconstructed")
        ax.plot(test_data_out[0,:,cnt], label="Original")
        ax.grid(True)
        cnt += 1

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.legend()
plt.show()

In [ ]:
recon_idx = [i for i in range(test_data_out.shape[2]) if i not in known_idx]

ss_res = np.sum((test_data_out - Y_pred) ** 2, axis=1)
ss_tot = np.sum((test_data_out - test_data_out.mean(axis=1, keepdims=True)) ** 2, axis=1)
r2_per_lead = 1.0 - ss_res / ss_tot  # (N, 12)

avg_r2 = r2_per_lead[:, recon_idx].mean(axis=1)

best_n = int(np.argmax(avg_r2))
worst_n = int(np.argmin(avg_r2))
print(f"Best  recording: idx={best_n:4d}  avg R2={avg_r2[best_n]:.4f}  {test_paths[best_n]}")
print(f"Worst recording: idx={worst_n:4d}  avg R2={avg_r2[worst_n]:.4f}  {test_paths[worst_n]}")


def plot_12lead(n, title):
    fig, axes = plt.subplots(4, 3, figsize=(20, 10))
    fig.suptitle(title, fontsize=16)
    cnt = 0
    for i in range(4):
        for j in range(3):
            ax = axes[i, j]
            ax.set_title(f"{lead_labels[cnt]}  (R2 = {r2_per_lead[n, cnt]:.3f})")
            ax.plot(Y_pred[n, :, cnt], label="Reconstructed")
            ax.plot(test_data_out[n, :, cnt], label="Original")
            ax.grid(True)
            cnt += 1
    axes[0, 0].legend(loc="upper right")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


plot_12lead(best_n,  f"Highest avg R2 - test recording {best_n} (avg R2 = {avg_r2[best_n]:.4f})")
plot_12lead(worst_n, f"Lowest avg R2 - test recording {worst_n} (avg R2 = {avg_r2[worst_n]:.4f})")